# hst_corpus_70m Cell Graph

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 21.02.2025
- Date of Last Modification: 12.03.2025 (by Sebastian Birk; sb75@sanger.ac.uk)

# 1. Setup

## 1.1 Load Libraries 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../../utils")

In [3]:
import logging
import os
import warnings
import pickle
import random
from datetime import datetime

import anndata as ad
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics
import seaborn as sns
import squidpy as sq
from datasets import load_from_disk
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split

from nichejepa.datasets import prepare_dataset
from nichejepa.infer import infer
from nichejepa.train import train
from nichejepa.utils.config import create_params_from_YAML_wandb_config
from nichejepa.utils.distributed import init_distributed
from nichejepa.utils.evaluation import clustering_metrics, classification_metrics

from analysis_utils import *

## 1.2 Load Config/Params
All setup will be done in the config file. Please edit that file to continue.

In [4]:
# Set params
dataset = 'hst_corpus_70m'
tokenizer = 'cell_graph'
load_timestamp = '04032025_000415_689'
read_checkpoint = 'nichejepa-ep5_4.pth.tar'
niche_key = 'niche_type'

In [5]:
model_config_folder_path = '../../config/model'
artifacts_folder_path = '../../artifacts'
datasets_folder_path = '/lustre/scratch126/cellgen/team361/DATASETS'
token_dict_path = '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl'

In [6]:
# Set up logger
rank=0
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

In [7]:
# Load config
if not load_timestamp:
    params = create_params_from_YAML_wandb_config(f'{model_config_folder_path}/{dataset}_{tokenizer}.yaml',logger)
else:
    params = create_params_from_YAML_wandb_config(f'{artifacts_folder_path}/{dataset}/{load_timestamp}/params.yaml',logger)

if read_checkpoint:
    params['state']['read_checkpoint'] = read_checkpoint
if load_timestamp:
    params['state']['load_checkpoint'] = True

params

INFO:root:Loaded parameters from YAML file.


{'data': {'batch_size': 228,
  'dataset_name': 'hst_corpus_70m',
  'n_segments': 11,
  'num_workers': 32,
  'pin_memory': True,
  'precomputed_n_nonzero_tokens': '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10_n_nonzero_tokens_train.pkl',
  'precomputed_split': '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/hst_corpus_70m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10_precomputed_split_train.pkl',
  'random_state': 0,
  'raw_data_folder_path': '/lustre/scratch126/cellgen/team361/DATASETS/silver/hst_corpus_70m',
  'sample_subset': False,
  'sampling_strategy': None,
  'seq_len_cell': 64,
  'seq_len_neighborhood': 640,
  'split': 0,
  'stratify': False,
  'token_dict_folder_path': '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl',
  'tokenized_data_folder_path': '/lustre/scratch126/cellgen/team361/DATASETS/gold/cell

In [8]:
#params['data']['batch_size'] = 64

In [9]:
# Define folder path for saving and loading artifacts
if not load_timestamp:
    current_timestamp = (
        datetime.now().strftime("%d%m%Y_%H%M%S") +
        f"_{datetime.now().microsecond // 1000:03d}")
    folder_path = os.path.join(artifacts_folder_path,
                               params['data']['dataset_name'],
                               current_timestamp)
else:
    folder_path = f'{artifacts_folder_path}/{dataset}/{load_timestamp}'
    
folder_path

'../../artifacts/hst_corpus_70m/04032025_000415_689'

## 1.3 Run Notebook Setup

In [10]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [11]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [12]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [13]:
sc.settings.figdir = f'{folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 4),
    facecolor='white',
    fontsize=10
)

# 2. Analysis

In [14]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'

## 2.1 Benchmarking Test Sets

### 2.1.1 chl59-8b_1p (NanoString CosMx Human NSCLC)

#### 2.1.1.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '59_batch0',
    #'59_batch3', # unannotated
    '59_batch6',
    '59_batch7',
    #'59_batch2', # unannotated
    '59_batch4',
    '59_batch1',
    '59_batch5']
params['data']['val_batch_ids'] = None

In [ ]:
# Set dataset params
dataset_name = 'chl59-8b_1p'
niche_key = 'niche'
batch_key = 'batch'
spot_sizes = [80] * len(params['data']['test_batch_ids'])

#### 2.1.1.2 Run Model Inference

In [ ]:
# Load data
_, _, test_dataset = prepare_dataset(params)
test_dataset

In [ ]:
# Run model inference
adata = infer(params,
              test_dataset,
              load_folder_path=folder_path,
              dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
              obs_cols=[niche_key],
              agg_type='avg',
              masked_tokens=None,
              agg_excluded_tokens=None,
              cell_gene_ids=[],
              neighborhood_gene_ids=[])

# Save adata
adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

In [ ]:
# Extract batch from cell ID
adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]

In [ ]:
# Remove missing ground truth
adata = adata[~adata.obs[niche_key].isna()]

#### 2.1.1.3 Plot Embedding Similarity Stats

In [ ]:
similarity_stats = compute_similarity_stats(
    adata,
    n_samples=None,
    type_key=niche_key,
    embedding_key=f'neighborhood_emb_layer_{layer}')

plot_similarity_heatmaps(
    similarity_stats,
    save_file_path=f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{niche_key}_cosinesimilarity.svg')

#### 2.1.1.4 Identify Niches

In [ ]:
# Compute UMAP representation for (aggregated) neighborhood gene embeddings
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep=f'neighborhood_emb_layer_{layer}',
                key_added='neighborhood')
sc.tl.umap(adata,
           neighbors_key='neighborhood')

In [ ]:
# Perform clustering until number of ground truth niches is matched
latent_leiden_resolution = 0.2
i = 0
n_niches = len(adata.obs[niche_key].value_counts() < 100)
while (i<25):
    latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'
    i += 1
    sc.tl.leiden(adata,
                 neighbors_key='neighborhood',
                 key_added=latent_cluster_key,
                 resolution=latent_leiden_resolution)

    n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
    print(f"Number of latent clusters {n_latent_clusters}")
    if n_latent_clusters == n_niches:
        break
    elif n_latent_clusters < (n_niches - 5):
        latent_leiden_resolution += 0.2
    elif n_latent_clusters < (n_niches - 3):
        latent_leiden_resolution += 0.1
    elif n_latent_clusters < n_niches:
        latent_leiden_resolution += 0.01
    elif n_latent_clusters > (n_niches + 5):
        latent_leiden_resolution -= 0.2
    elif n_latent_clusters > (n_niches + 3):
        latent_leiden_resolution -= 0.1
    else:
        latent_leiden_resolution -= 0.01

# Compute metrics and store to disk
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari = adjusted_rand_score(labels, labels_pred)
print(f'NMI is {nmi}. ARI is {ari}.')
metrics_df = pd.DataFrame({
    'metric': ['NMI', 'ARI'],
    'value': [nmi, ari]
})
metrics_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_metrics.csv',
                  index=False)

# Plot UMAPs
os.makedirs(f'{folder_path}/figures', exist_ok=True)
labels = ['batch', niche_key, latent_cluster_key]
# Plot (aggregated) neighborhood gene embeddings, colored by labels
for label in labels:
    sc.pl.umap(adata,
               color=label,
               show=False)
    plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{label}_umap.svg',
                dpi=300,
                bbox_inches='tight')

# Plot niches in tissue
batch_ids = adata.obs['batch'].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs['batch'] == batch_id]
    sc.pl.spatial(adata_batch,
                  color=latent_cluster_key,
                  palette=custom_palette,
                  spot_size=spot_sizes[i],
                  save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg')

### 2.1.2 chr78-11b_1p (ISS Cartana Human Reproductive Tract)

#### 2.1.2.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '78_batch3',
    '78_batch10',
    '78_batch1',
    '78_batch0',
    '78_batch2',
    '78_batch6',
    '78_batch7',
    '78_batch9',
    '78_batch8',
    '78_batch4',
    '78_batch5']
params['data']['val_batch_ids'] = None

In [ ]:
# Set dataset params
dataset_name = 'chr78-11b_1p'
niche_key = 'broad_anatomy'
batch_key = 'batch'
spot_sizes = [1920 * 0.093] * len(params['data']['test_batch_ids'])

#### 2.1.2.2 Run Model Inference

In [ ]:
# Load data
_, _, test_dataset = prepare_dataset(params)
test_dataset

In [ ]:
# Run model inference
adata = infer(params,
              test_dataset,
              load_folder_path=folder_path,
              dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
              obs_cols=[niche_key],
              agg_type='avg',
              masked_tokens=None,
              agg_excluded_tokens=None,
              cell_gene_ids=[],
              neighborhood_gene_ids=[])

# Save adata
adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

In [ ]:
# Extract batch from cell ID
adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]

In [ ]:
 # Remove missing ground truth
adata = adata[~adata.obs[niche_key].isna()]

#### 2.1.2.3 Plot Embedding Similarity Stats

In [ ]:
similarity_stats = compute_similarity_stats(
    adata,
    n_samples=None,
    type_key=niche_key,
    embedding_key=f'neighborhood_emb_layer_{layer}')

plot_similarity_heatmaps(
    similarity_stats,
    save_file_path=f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{niche_key}_cosinesimilarity.svg')

#### 2.1.2.4 Identify Niches

In [ ]:
# Compute UMAP representation for (aggregated) neighborhood gene embeddings
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep=f'neighborhood_emb_layer_{layer}',
                key_added='neighborhood')
sc.tl.umap(adata,
           neighbors_key='neighborhood')

In [ ]:
# Perform clustering until number of ground truth niches is matched
latent_leiden_resolution = 0.2
i = 0
n_niches = len(adata.obs[niche_key].value_counts() < 100)
while (i<25):
    latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'
    i += 1
    sc.tl.leiden(adata,
                 neighbors_key='neighborhood',
                 key_added=latent_cluster_key,
                 resolution=latent_leiden_resolution)

    n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
    print(f"Number of latent clusters {n_latent_clusters}")
    if n_latent_clusters == n_niches:
        break
    elif n_latent_clusters < (n_niches - 5):
        latent_leiden_resolution += 0.02
    elif n_latent_clusters < (n_niches - 3):
        latent_leiden_resolution += 0.01
    elif n_latent_clusters < n_niches:
        latent_leiden_resolution += 0.001
    elif n_latent_clusters > (n_niches + 5):
        latent_leiden_resolution -= 0.02
    elif n_latent_clusters > (n_niches + 3):
        latent_leiden_resolution -= 0.01
    else:
        latent_leiden_resolution -= 0.001

# Compute metrics and store to disk
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari = adjusted_rand_score(labels, labels_pred)
print(f'NMI is {nmi}. ARI is {ari}.')
metrics_df = pd.DataFrame({
    'metric': ['NMI', 'ARI'],
    'value': [nmi, ari]
})
metrics_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_metrics.csv',
                  index=False)

# Plot UMAPs
os.makedirs(f'{folder_path}/figures', exist_ok=True)
labels = ['batch', niche_key, latent_cluster_key]
# Plot (aggregated) neighborhood gene embeddings, colored by labels
for label in labels:
    sc.pl.umap(adata,
               color=label,
               show=False)
    plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{label}_umap.svg',
                dpi=300,
                bbox_inches='tight')

# Plot niches in tissue
batch_ids = adata.obs['batch'].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs['batch'] == batch_id]
    sc.pl.spatial(adata_batch,
                  color=latent_cluster_key,
                  palette=custom_palette,
                  spot_size=spot_sizes[i],
                  save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg')

### 2.1.3 xhs49-1b_1p (Xenium Human Melanoma)

#### 2.1.2.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '49_batch0']
params['data']['val_batch_ids'] = None

In [ ]:
# Set dataset params
dataset_name = 'xhs49-1b_1p'
niche_key = 'niche_name'
spot_sizes = [20] * len(params['data']['test_batch_ids'])

#### 2.1.3.2 Run Model Inference

In [ ]:
# Load data
_, _, test_dataset = prepare_dataset(params)
test_dataset

In [ ]:
# Run model inference
adata = infer(params,
              test_dataset,
              load_folder_path=folder_path,
              dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
              obs_cols=[niche_key],
              agg_type='avg',
              masked_tokens=None,
              agg_excluded_tokens=None,
              cell_gene_ids=[],
              neighborhood_gene_ids=[])

# Save adata
adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

In [ ]:
# Extract batch from cell ID
adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]

In [ ]:
# Remove missing ground truth
adata = adata[~adata.obs[niche_key].isna()]

#### 2.1.3.3 Plot Embedding Similarity Stats

In [ ]:
similarity_stats = compute_similarity_stats(
    adata,
    n_samples=None,
    type_key=niche_key,
    embedding_key=f'neighborhood_emb_layer_{layer}')

plot_similarity_heatmaps(
    similarity_stats,
    save_file_path=f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{niche_key}_cosinesimilarity.svg')

#### 2.1.3.4 Identify Niches

In [ ]:
# Compute UMAP representation for (aggregated) neighborhood gene embeddings
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep=f'neighborhood_emb_layer_{layer}',
                key_added='neighborhood')
sc.tl.umap(adata,
           neighbors_key='neighborhood')

In [ ]:
# Perform clustering until number of ground truth niches is matched
latent_leiden_resolution = 0.2
i = 0
n_niches = len(adata.obs[niche_key].value_counts() < 100)
while (i<25):
    latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'
    i += 1
    sc.tl.leiden(adata,
                 neighbors_key='neighborhood',
                 key_added=latent_cluster_key,
                 resolution=latent_leiden_resolution)

    n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
    print(f"Number of latent clusters {n_latent_clusters}")
    if n_latent_clusters == n_niches:
        break
    elif n_latent_clusters < (n_niches - 5):
        latent_leiden_resolution += 0.02
    elif n_latent_clusters < (n_niches - 3):
        latent_leiden_resolution += 0.01
    elif n_latent_clusters < n_niches:
        latent_leiden_resolution += 0.001
    elif n_latent_clusters > (n_niches + 5):
        latent_leiden_resolution -= 0.02
    elif n_latent_clusters > (n_niches + 3):
        latent_leiden_resolution -= 0.01
    else:
        latent_leiden_resolution -= 0.001
        
# Compute metrics and store to disk
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari = adjusted_rand_score(labels, labels_pred)
print(f'NMI is {nmi}. ARI is {ari}.')
metrics_df = pd.DataFrame({
    'metric': ['NMI', 'ARI'],
    'value': [nmi, ari]
})
metrics_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_metrics.csv',
                  index=False)

# Plot UMAPs
os.makedirs(f'{folder_path}/figures', exist_ok=True)
labels = ['batch', niche_key, latent_cluster_key]
# Plot (aggregated) neighborhood gene embeddings, colored by labels
for label in labels:
    sc.pl.umap(adata,
               color=label,
               show=False)
    plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{label}_umap.svg',
                dpi=300,
                bbox_inches='tight')

# Plot niches in tissue
batch_ids = adata.obs['batch'].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs['batch'] == batch_id]
    sc.pl.spatial(adata_batch,
                  color=latent_cluster_key,
                  palette=custom_palette,
                  spot_size=spot_sizes[i],
                  save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg')

### 2.1.4 xhs1000-39b_1p (Xenium Human Skin - Eczema)

#### 2.1.4.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '1000_batch11',
    '1000_batch19',
    '1000_batch32']
params['data']['val_batch_ids'] = None

In [ ]:
# Set dataset params
dataset_name = 'xhs1000-39b_1p'
niche_key = 'niche_type'
batch_key = 'sample'
spot_sizes = [20] * len(params['data']['test_batch_ids'])

#### 2.1.4.2 Run Model Inference

In [ ]:
# Load data
_, _, test_dataset = prepare_dataset(params)
test_dataset

In [ ]:
# Run model inference
adata = infer(params,
              test_dataset,
              load_folder_path=folder_path,
              dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
              obs_cols=[niche_key],
              agg_type='avg',
              masked_tokens=None,
              agg_excluded_tokens=None,
              cell_gene_ids=[],
              neighborhood_gene_ids=[])

# Save adata
adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

In [ ]:
# Extract batch from cell ID
adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]

In [ ]:
# Remove missing ground truth
adata = adata[~adata.obs[niche_key].isna()]

#### 2.1.4.3 Plot Embedding Similarity Stats

In [ ]:
similarity_stats = compute_similarity_stats(
    adata,
    n_samples=None,
    type_key=niche_key,
    embedding_key=f'neighborhood_emb_layer_{layer}')

plot_similarity_heatmaps(
    similarity_stats,
    save_file_path=f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{niche_key}_cosinesimilarity.svg')

#### 2.1.4.4 Identify Niches

In [ ]:
# Compute UMAP representation for (aggregated) neighborhood gene embeddings
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep=f'neighborhood_emb_layer_{layer}',
                key_added='neighborhood')
sc.tl.umap(adata,
           neighbors_key='neighborhood')

In [ ]:
# Perform clustering until number of ground truth niches is matched
latent_leiden_resolution = 0.2
i = 0
n_niches = len(adata.obs[niche_key].value_counts() < 100)
while (i<25):
    latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'
    i += 1
    sc.tl.leiden(adata,
                 neighbors_key='neighborhood',
                 key_added=latent_cluster_key,
                 resolution=latent_leiden_resolution)

    n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
    print(f"Number of latent clusters {n_latent_clusters}")
    if n_latent_clusters == n_niches:
        break
    elif n_latent_clusters < (n_niches - 5):
        latent_leiden_resolution += 0.02
    elif n_latent_clusters < (n_niches - 3):
        latent_leiden_resolution += 0.01
    elif n_latent_clusters < n_niches:
        latent_leiden_resolution += 0.001
    elif n_latent_clusters > (n_niches + 5):
        latent_leiden_resolution -= 0.02
    elif n_latent_clusters > (n_niches + 3):
        latent_leiden_resolution -= 0.01
    else:
        latent_leiden_resolution -= 0.001

# Compute metrics and store to disk
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari = adjusted_rand_score(labels, labels_pred)
print(f'NMI is {nmi}. ARI is {ari}.')
metrics_df = pd.DataFrame({
    'metric': ['NMI', 'ARI'],
    'value': [nmi, ari]
})
metrics_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_metrics.csv',
                  index=False)

# Plot UMAPs
os.makedirs(f'{folder_path}/figures', exist_ok=True)
labels = ['batch', niche_key, latent_cluster_key]
# Plot (aggregated) neighborhood gene embeddings, colored by labels
for label in labels:
    sc.pl.umap(adata,
               color=label,
               show=False)
    plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{label}_umap.svg',
                dpi=300,
                bbox_inches='tight')

# Plot niches in tissue
batch_ids = adata.obs['batch'].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs['batch'] == batch_id]
    sc.pl.spatial(adata_batch,
                  color=latent_cluster_key,
                  palette=custom_palette,
                  spot_size=spot_sizes[i],
                  save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg')

## 2.2 Full Corpus

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

In [ ]:
# Load dataset
dataset = load_from_disk(params['data']['tokenized_data_folder_path'])

In [ ]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
latent_leiden_resolution = 0.5 # 0.1, 0.5
latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'

In [ ]:
# Subset dataset
step_size = 2000000
steps = int(np.ceil(len(dataset)/step_size))

dataset_subsets = []

for i in range(steps):
    if ((i+1)*step_size) > len(dataset):
        end_step = len(dataset)
    else:
        end_step = (i+1)*step_size
    dataset_subsets.append(dataset.select(np.arange(i*step_size, end_step)))

del dataset

In [ ]:
# Get batches for current dataset subset
# Iteration 6 might need 600k RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 18:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    if i != 6:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    if i >= 10:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 10:
        continue
    if i >= 20:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 20:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 17:
        continue
    if i >= 20:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

## 2.3 Analysis Data

In [15]:
batch_ids = ['1000', '1007', '1008']

In [16]:
# Load dataset
dataset = load_from_disk(params['data']['tokenized_data_folder_path'])

Loading dataset from disk:   0%|          | 0/32 [00:00<?, ?it/s]

In [17]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
latent_leiden_resolution = 0.5 # 0.1, 0.5
latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'

In [18]:
dataset_subset = dataset.filter(lambda x: x['cell_id'].split('_')[0].isin(batch_ids))

Filter:   0%|          | 0/70122697 [00:00<?, ? examples/s]

AttributeError: 'str' object has no attribute 'isin'

In [ ]:
dataset_subset